# Sinhala Document OCR — Local Pipeline

**For testing only (recommended):** leave train flags `False`, set `TEST_IMAGE_PATH` or use the file picker, then **Kernel → Restart & Run All**.

One general model: `models/crnn_best.pth`. Real poem lines are mixed into general training (not a separate finetune stage).

## Table of contents

| # | Section | Skip when |
|---|---------|-----------|
| 1 | Setup | — |
| 2 | Install (optional) | `RUN_INSTALL=False` |
| 3 | Fonts | — |
| 4 | **Config & flags** (one control cell) | — |
| 5 | Generate synthetic lines | data already present |
| 6 | Generate page-synth data | data already present |
| 7 | Train general CRNN | `crnn_best.pth` exists |
| 8 | **Test with real image** | — |
| 9 | Poem CER self-check (optional) | no poem crops |
| 10 | Save debug (optional) | — |


## 1. Setup


In [ ]:
import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "local_pipeline.ipynb").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = NOTEBOOK_DIR
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import torch
from src.utils.display import configure_display_utf8, setup_matplotlib_sinhala

configure_display_utf8()
print("REPO_ROOT:", REPO_ROOT)
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")


## 2. Install dependencies (optional)


In [ ]:
RUN_INSTALL = False  # True once on a fresh environment

if RUN_INSTALL:
    %pip install -q -r requirements.txt
else:
    print("Skipping pip install (set RUN_INSTALL=True on a new environment).")


## 3. Fonts


In [ ]:
from pathlib import Path

FONT_CANDIDATES = [
    Path(r"C:/Windows/Fonts/Nirmala.ttc"),
    Path(r"C:/Windows/Fonts/iskpota.ttf"),
    REPO_ROOT / "fonts" / "NotoSansSinhala-Regular.ttf",
    REPO_ROOT / "fonts" / "NotoSerifSinhala-Regular.ttf",
    REPO_ROOT / "fonts" / "AbhayaLibre-Regular.ttf",
]
FONT_PATHS = [str(p) for p in FONT_CANDIDATES if p.is_file()]
setup_matplotlib_sinhala()
if FONT_PATHS:
    print("Fonts ready:", len(FONT_PATHS))
    for fp in FONT_PATHS[:5]:
        print(" ", Path(fp).name)
else:
    print("WARNING: No Sinhala fonts found. Run scripts/download_fonts.ps1 or install Nirmala UI.")


## 4. Config & flags — one control cell

Edit here, then **Restart & Run All**.

- Testing: leave `RUN_GENERATE*` / `RUN_TRAIN` False; set `TEST_IMAGE_PATH` or leave empty for the picker.
- First-time setup: set generate/train True once (auto-skips later when artifacts exist).


In [ ]:
from pathlib import Path
import torch
from src.utils.common import load_config, get_logger, get_device
from src.recognition.inference import inference_options_from_config

logger = get_logger("local_pipeline")
CFG_PATH = REPO_ROOT / "configs" / "local.yaml"
cfg = load_config(str(CFG_PATH))
INF_OPTS = inference_options_from_config(cfg)
DEVICE = get_device(cfg["train"].get("device", "auto"))

# === ONE CONTROL CELL ===
RUN_INSTALL = False          # mirrored in section 2; keep False after first env setup
RUN_GENERATE = False         # synthetic line crops
RUN_GENERATE_PAGES = False   # detector-in-the-loop page crops
RUN_TRAIN = False            # train/continue general model -> models/crnn_best.pth
RUN_POEM_CER = True          # optional self-check on Kanyawee crops
RUN_SAVE_DEBUG = True

# Real-image test (section 8). Empty string -> file picker, then demo fallback.
TEST_IMAGE_PATH = ""  # e.g. r"data/uploads/test2.png" or an absolute path
USE_FILE_PICKER = True       # if TEST_IMAGE_PATH empty

# Auto-skip: force generate/train only when artifacts are missing
SYN_DIR = REPO_ROOT / cfg["paths"]["synthetic_dir"]
PAGES_DIR = REPO_ROOT / cfg["paths"].get("synthetic_pages_dir", "data/synthetic_pages")
TRAIN_LABELS = SYN_DIR / "train_labels.txt"
PAGES_TRAIN_LABELS = PAGES_DIR / "train_labels.txt"
POEM_LABELS = REPO_ROOT / "data" / "real" / "labels" / "poem_kanyawee.txt"
POEM_AUG_LABELS = REPO_ROOT / "data" / "real" / "labels" / "poem_kanyawee_aug.txt"
MODELS_DIR = REPO_ROOT / cfg["paths"]["models_dir"]
CHARSET_PATH = REPO_ROOT / cfg["paths"]["charset_path"]
CKPT_PATH = MODELS_DIR / "crnn_best.pth"
UPLOADS_DIR = REPO_ROOT / cfg["paths"]["uploads_dir"]
DEBUG_DIR = REPO_ROOT / cfg["paths"]["debug_dir"]

DEMO_CANDIDATES = [
    UPLOADS_DIR / "test2.png",
    REPO_ROOT / "data" / "eval_pages" / "adversarial" / "adv_poem.png",
    REPO_ROOT / "data" / "eval_pages" / "realistic" / "page_01.png",
]

if not TRAIN_LABELS.is_file():
    RUN_GENERATE = True
    print("Auto: RUN_GENERATE=True (missing train_labels.txt)")
if not PAGES_TRAIN_LABELS.is_file():
    RUN_GENERATE_PAGES = True
    print("Auto: RUN_GENERATE_PAGES=True (missing page train labels)")
if not CKPT_PATH.is_file():
    RUN_TRAIN = True
    print("Auto: RUN_TRAIN=True (missing crnn_best.pth)")

NUM_SAMPLES = 30000 if torch.cuda.is_available() else 5000
NUM_PAGES = 4000 if torch.cuda.is_available() else 300
TRAIN_EPOCHS = int(cfg["train"].get("epochs", 40))

print("config:", CFG_PATH)
print("checkpoint:", CKPT_PATH.name, "| exists:", CKPT_PATH.is_file())
print("device:", DEVICE, "| height:", INF_OPTS["height"], "| decode:", INF_OPTS.get("decode"))
print("flags: GENERATE=%s PAGES=%s TRAIN=%s" % (RUN_GENERATE, RUN_GENERATE_PAGES, RUN_TRAIN))
print("TEST_IMAGE_PATH:", TEST_IMAGE_PATH or "(picker / demo)")


## 5. Generate synthetic line data (optional)


In [ ]:
import subprocess

if not RUN_GENERATE and TRAIN_LABELS.is_file():
    n = sum(1 for _ in open(TRAIN_LABELS, encoding="utf-8"))
    print(f"SKIP generate lines — {TRAIN_LABELS.name} has {n} rows")
elif not RUN_GENERATE:
    print("SKIP generate lines (RUN_GENERATE=False and no labels)")
else:
    cmd = [
        sys.executable, str(REPO_ROOT / "scripts" / "generate_data.py"),
        "--config", str(CFG_PATH), "--large", "--num", str(NUM_SAMPLES),
    ]
    print("Running:", " ".join(cmd))
    rc = subprocess.call(cmd, cwd=str(REPO_ROOT))
    if rc != 0:
        raise RuntimeError(f"generate_data.py failed with code {rc}")
    print("Synthetic data ready:", TRAIN_LABELS)


## 6. Generate detector-in-the-loop page data (optional)


In [ ]:
if not RUN_GENERATE_PAGES and PAGES_TRAIN_LABELS.is_file():
    n = sum(1 for _ in open(PAGES_TRAIN_LABELS, encoding="utf-8"))
    print(f"SKIP page generate — {PAGES_TRAIN_LABELS.name} has {n} rows")
elif not RUN_GENERATE_PAGES:
    print("SKIP page generate (RUN_GENERATE_PAGES=False)")
else:
    cmd = [
        sys.executable,
        str(REPO_ROOT / "scripts" / "generate_pages.py"),
        "--config", str(CFG_PATH),
        "--num-pages", str(NUM_PAGES),
    ]
    print("Running:", " ".join(cmd))
    rc = __import__("subprocess").call(cmd, cwd=str(REPO_ROOT))
    if rc != 0:
        raise RuntimeError(f"generate_pages.py failed with code {rc}")
    print("Page supplement ready:", PAGES_TRAIN_LABELS)


## 7. Train general CRNN (optional)

Trains / continues `models/crnn_best.pth` with synthetic + page-synth + (if present) augmented poem lines.
Skipped when the checkpoint exists unless you set `RUN_TRAIN=True`.


In [ ]:
import subprocess

if not RUN_TRAIN and CKPT_PATH.is_file():
    print(f"SKIP train — using existing {CKPT_PATH}")
elif not RUN_TRAIN:
    print("SKIP train (RUN_TRAIN=False and no checkpoint)")
else:
    if not TRAIN_LABELS.is_file():
        raise FileNotFoundError("Missing synthetic train labels; set RUN_GENERATE=True")
    # Prefer mix_real when poem aug labels exist; else plain local.yaml
    mix_cfg = REPO_ROOT / "configs" / "mix_real.yaml"
    train_cfg = mix_cfg if (mix_cfg.is_file() and POEM_AUG_LABELS.is_file()) else CFG_PATH
    cmd = [sys.executable, "-m", "src.recognition.train", "--config", str(train_cfg)]
    if PAGES_TRAIN_LABELS.is_file():
        cmd += ["--extra-labels", str(PAGES_TRAIN_LABELS)]
    if POEM_AUG_LABELS.is_file():
        cmd += ["--extra-labels", str(POEM_AUG_LABELS)]
    elif POEM_LABELS.is_file():
        cmd += ["--extra-labels", str(POEM_LABELS)]
    if CKPT_PATH.is_file():
        cmd += ["--resume", str(CKPT_PATH)]
    print("Running:", " ".join(cmd))
    rc = subprocess.call(cmd, cwd=str(REPO_ROOT))
    if rc != 0:
        raise RuntimeError(f"Training failed with code {rc}")
    print("Training done:", CKPT_PATH)


## 8. Test with a real image

Detect lines → recognize with `crnn_best.pth` → show crops + Sinhala predictions + full transcription.

Resolution order: `TEST_IMAGE_PATH` → file picker (if enabled) → bundled demo under `data/uploads` / `data/eval_pages`.


In [ ]:
from pathlib import Path
import cv2
import matplotlib.pyplot as plt
import torch

from src.charset import Charset
from src.detection.text_detection import build_detector
from src.evaluation.pipeline_eval import run_pipeline_on_image_path
from src.recognition.model import build_crnn
from src.utils.common import load_checkpoint

def _resolve_test_image() -> Path:
    if TEST_IMAGE_PATH:
        p = Path(TEST_IMAGE_PATH)
        if not p.is_absolute():
            p = REPO_ROOT / p
        if p.is_file():
            return p
        raise FileNotFoundError(f"TEST_IMAGE_PATH not found: {p}")
    if USE_FILE_PICKER:
        try:
            from tkinter import Tk, filedialog
            root = Tk(); root.withdraw(); root.attributes("-topmost", True)
            chosen = filedialog.askopenfilename(
                title="Select a Sinhala document image",
                filetypes=[("Images", "*.png;*.jpg;*.jpeg;*.tif;*.bmp"), ("All", "*.*")],
            )
            root.destroy()
            if chosen:
                return Path(chosen)
        except Exception as exc:
            print("File picker unavailable:", exc)
    for cand in DEMO_CANDIDATES:
        if cand.is_file():
            print("Using demo image:", cand)
            return cand
    raise FileNotFoundError(
        "No test image. Set TEST_IMAGE_PATH, use the picker, or place a file in data/uploads/."
    )

if not CKPT_PATH.is_file():
    raise FileNotFoundError(f"Missing {CKPT_PATH}. Set RUN_TRAIN=True or train first.")
if not CHARSET_PATH.is_file():
    raise FileNotFoundError(f"Missing {CHARSET_PATH}")

test_image = _resolve_test_image()
print("Test image:", test_image)

charset = Charset.load(str(CHARSET_PATH))
model = build_crnn(charset.num_classes, cfg.get("model"), in_channels=cfg["image"]["channels"]).to(DEVICE)
load_checkpoint(str(CKPT_PATH), model, map_location=str(DEVICE))
model.eval()
print("Loaded", CKPT_PATH.name, "| height", INF_OPTS["height"])

det_cfg = dict(cfg.get("detection", {}))
detector = build_detector(det_cfg)
result = run_pipeline_on_image_path(
    model, charset, str(test_image), detector, INF_OPTS, det_cfg, DEVICE,
)
boxes, crops, texts = result["boxes"], result["crops"], result["display_texts"]
raw_texts = result["texts"]
print(f"Detected {len(boxes)} line(s) with method={det_cfg.get('method', 'projection')}")
print("--- Full transcription ---")
print("\n".join(texts) if texts else "(no lines)")
print("--------------------------")

# Overview
bgr = result["bgr"]
vis = bgr.copy()
for (x, y, w, h) in boxes:
    cv2.rectangle(vis, (x, y), (x + w, y + h), (0, 180, 0), 2)
plt.figure(figsize=(10, 10))
plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))
plt.title(f"{test_image.name} — {len(boxes)} lines")
plt.axis("off")
plt.show()

# Per-line crops + predictions
n_show = min(len(crops), 20)
for i in range(n_show):
    fig, ax = plt.subplots(1, 1, figsize=(12, 1.6))
    ax.imshow(crops[i], cmap="gray")
    ax.set_title(f"L{i+1:02d}: {texts[i]}", fontsize=12)
    ax.axis("off")
    plt.show()

upload_predictions = list(raw_texts)
UPLOADED_IMAGE_PATH = str(test_image)


## 9. Poem CER self-check (optional)

Scores `crnn_best.pth` on Kanyawee line crops vs `data/real/labels/poem_kanyawee.txt`.
If those lines were mixed into training, treat this as **train-set** poem CER (still useful for the demo).


In [ ]:
from src.data.dataset import read_labels
from src.evaluation.metrics import cer, corpus_cer
from src.recognition.predict import predict_image

if not RUN_POEM_CER:
    print("SKIP poem CER (RUN_POEM_CER=False)")
elif not POEM_LABELS.is_file():
    print("SKIP poem CER — missing", POEM_LABELS)
else:
    rows = read_labels(str(POEM_LABELS))
    preds, gts = [], []
    print(f"Evaluating {CKPT_PATH.name} on {len(rows)} poem lines")
    for rel, gt in rows:
        path = REPO_ROOT / "data" / "real" / rel
        if not path.is_file():
            print(" missing", path)
            continue
        pred = predict_image(
            model, charset, str(path),
            INF_OPTS["height"], INF_OPTS["max_width"], INF_OPTS["channels"], DEVICE,
            auto_invert=INF_OPTS["auto_invert"], denoise=INF_OPTS["denoise"],
            min_model_width=INF_OPTS.get("min_model_width", 0),
            pad_to_height=INF_OPTS.get("pad_to_height", True),
            warn_garbage=False,
            decode_mode=INF_OPTS.get("decode", "greedy"),
        )
        c = cer(gt, pred)
        preds.append(pred); gts.append(gt)
        print(f"{path.name}  CER={c:.3f}")
        print(f"  GT:   {gt}")
        print(f"  PRED: {pred}")
    if gts:
        print(f"\nPoem corpus CER: {corpus_cer(gts, preds):.4f}")


## 10. Optional — save debug outputs


In [ ]:
import json
from datetime import datetime, timezone
from src.evaluation.pipeline_eval import save_debug

if not RUN_SAVE_DEBUG:
    print("SKIP debug save")
elif not upload_predictions:
    print("SKIP debug save — no predictions from section 8")
else:
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    out = DEBUG_DIR / f"run_{ts}"
    payload = {
        "source_image": UPLOADED_IMAGE_PATH,
        "checkpoint": CKPT_PATH.name,
        "num_lines": len(upload_predictions),
        "lines": [{"line": i + 1, "text": t} for i, t in enumerate(upload_predictions)],
    }
    save_debug(str(out), result["bgr"], boxes, crops, upload_predictions, extra=payload)
    print("Saved debug to", out)
